In [ ]:
# Cell 1: Kết nối Google Drive & giải nén, cài đặt thư viện
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U albumentations

import os, cv2, random, shutil, glob, gc, json, time
import albumentations as A
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import psutil
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

zip_path = "/content/drive/MyDrive/PBL5/Dataset/dataset.zip"
extract_path = "/content/dataset"
if not os.path.exists(extract_path):
    !unzip -q "{zip_path}" -d "{extract_path}"

In [ ]:
# Cell 2: Crop BBox theo YOLO label và giữ nguyên Split 78:11:11
class_mapping = {0: "broken", 1: "defect", 2: "whole"}
PADDING = 20

def crop_by_yolo_label(image_dir, label_dir, output_dir, padding=PADDING):
    for name in class_mapping.values():
        os.makedirs(os.path.join(output_dir, name), exist_ok=True)
    crop_count = {name: 0 for name in class_mapping.values()}

    for img_file in os.listdir(image_dir):
        if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
            continue
        base_name = os.path.splitext(img_file)[0]
        img_path = os.path.join(image_dir, img_file)

        label_file = os.path.join(label_dir, base_name + ".txt")
        lines = []

        if os.path.exists(label_file):
            with open(label_file, 'r') as f:
                lines = f.readlines()

        # Bỏ qua nếu file trống (background)
        if len(lines) == 0:
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            class_name = class_mapping.get(class_id, "unknown")
            if class_name == "unknown":
                continue

            xc, yc = float(parts[1]) * img_w, float(parts[2]) * img_h
            bw, bh = float(parts[3]) * img_w, float(parts[4]) * img_h
            x_min = max(0, int(xc - bw / 2) - padding)
            y_min = max(0, int(yc - bh / 2) - padding)
            x_max = min(img_w, int(xc + bw / 2) + padding)
            y_max = min(img_h, int(yc + bh / 2) + padding)

            cropped = img[y_min:y_max, x_min:x_max]
            if cropped.size == 0:
                continue
            save_path = os.path.join(output_dir, class_name, f"{base_name}_crop_{i}.jpg")
            cv2.imwrite(save_path, cropped)
            crop_count[class_name] += 1

    return crop_count

cropped_base_dir = "/content/cropped_datasets"
for split in ["train", "valid", "test"]:
    print(f"Crop tập {split.upper()}...")
    out_dir = os.path.join(cropped_base_dir, split)
    count = crop_by_yolo_label(
        image_dir=f"/content/dataset/{split}/images",
        label_dir=f"/content/dataset/{split}/labels",
        output_dir=out_dir,
    )
    print(f"  {split}: {count}")

In [ ]:
# Cell 3: Khởi tạo biến từ các tập đã chia sẵn
K_FOLDS = 5
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TARGET_COUNT = 500
AUTOTUNE = tf.data.AUTOTUNE
class_mapping = {0: "broken", 1: "defect", 2: "whole"}

def load_split_info(split_name):
    images = []
    labels = []
    split_dir = os.path.join(cropped_base_dir, split_name)
    for class_name in class_mapping.values():
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.exists(class_dir): continue
        for f in os.listdir(class_dir):
            if f.endswith((".jpg", ".png")):
                images.append(os.path.join(class_dir, f))
                labels.append(class_name)
    return np.array(images), np.array(labels)

train_images, train_labels = load_split_info("train")
valid_images, valid_labels = load_split_info("valid")
test_images, test_labels = load_split_info("test")

label_to_idx = {v: k for k, v in class_mapping.items()}
train_labels_int = np.array([label_to_idx[l] for l in train_labels])
valid_labels_int = np.array([label_to_idx[l] for l in valid_labels])
test_labels_int = np.array([label_to_idx[l] for l in test_labels])

# --- TẠO POOL 89% CHO HYBRID CV ---
pool_images = np.concatenate([train_images, valid_images])
pool_labels = np.concatenate([train_labels, valid_labels])
pool_labels_int = np.array([label_to_idx[l] for l in pool_labels])

print(f"\n{'='*50}")
print(f"ĐỒNG BỘ DATA SPLIT 78:11:11 HOÀN TẤT:")
print(f"   Tập Train (78%): {len(train_images)} ảnh")
print(f"   Tập Valid (11%): {len(valid_images)} ảnh")
print(f"   Tập Test  (11%): {len(test_images)} ảnh")
print(f"{'='*50}")
print(f"   -> Đã tạo Pool 89% ({len(pool_images)} ảnh) dùng cho K-Fold CV.")
for cls in class_mapping.values():
    print(f"   - {cls}: Train={sum(train_labels==cls)}, Valid={sum(valid_labels==cls)}, Test={sum(test_labels==cls)}")

In [ ]:
# Cell 4: Khai báo Augmentation Pipeline
aug_transform = A.Compose([
    A.Rotate(limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(
        hue_shift_limit=3,
        sat_shift_limit=8,
        val_shift_limit=8,
        p=0.4
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(std_range=(0.01, 0.03), mean_range=(0, 0), p=0.2),
])

def augment(src_dir, dst_dir, target_count):
    os.makedirs(dst_dir, exist_ok=True)
    original_images = glob.glob(os.path.join(src_dir, "*.jpg"))
    if len(original_images) == 0:
        print(f"Warning: No images found in {src_dir}")
        return

    for img_path in original_images:
        shutil.copy(img_path, dst_dir)

    current = len(original_images)
    needed = target_count - current
    if needed > 0:
        for i in range(needed):
            src_path = random.choice(original_images)
            img = cv2.imread(src_path)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            augmented = aug_transform(image=img_rgb)
            aug_bgr = cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(dst_dir, f"aug_{i}_{os.path.basename(src_path)}"), aug_bgr)

print("Augmentation pipeline đã sẵn sàng.")

In [ ]:
# Cell 5: Kiến trúc Model MobileNetV2 (3 Classes)
def se_block(input_tensor, ratio=16):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu',
                      kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid',
                      kernel_initializer='he_normal', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

def build_mobilenetv2(num_classes=3, dropout_rate=0.3, l2_lambda=1e-4,
                      hidden_units=128):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = layers.Rescaling(scale=1./127.5, offset=-1.0)(inputs)
    x = base_model(x, training=False)
    x = se_block(x, ratio=16)  # SE ratio cố định = 16

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(hidden_units,
                     kernel_regularizer=regularizers.l2(l2_lambda))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(
        num_classes, activation='softmax',
        kernel_regularizer=regularizers.l2(l2_lambda)
    )(x)

    model = Model(inputs, outputs)
    return model

model_test = build_mobilenetv2(num_classes=3)
model_test.summary()
del model_test
tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# Cell 6: Helper Functions cho K-Fold
def sample_head_params(n_trials, seed=42):
    rng = np.random.RandomState(seed)
    trials = []
    for _ in range(n_trials):
        trials.append({
            'lr_phase1':    float(10 ** rng.uniform(-3.5, -2.5)),
            'dropout_rate': round(float(rng.uniform(0.2, 0.4)), 3),
            'hidden_units': int(rng.choice([128, 256])),
        })
    return trials

def prepare_fold_data(fold_idx, train_indices, val_indices):
    # Dùng pool_images thay vì train_images cho Hybrid CV
    fold_train_imgs = pool_images[train_indices]
    fold_train_lbls = pool_labels[train_indices]
    fold_val_imgs   = pool_images[val_indices]
    fold_val_lbls   = pool_labels[val_indices]

    fold_dir = f"/content/fold_{fold_idx}"
    fold_train_dir = os.path.join(fold_dir, "train")
    fold_val_dir   = os.path.join(fold_dir, "val")
    if os.path.exists(fold_dir): shutil.rmtree(fold_dir)

    for img_path, label in zip(fold_val_imgs, fold_val_lbls):
        dst = os.path.join(fold_val_dir, label)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(img_path, dst)

    fold_train_raw = os.path.join(fold_dir, "train_raw")
    for img_path, label in zip(fold_train_imgs, fold_train_lbls):
        dst = os.path.join(fold_train_raw, label)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(img_path, dst)

    for class_name in class_mapping.values():
        augment(
            src_dir=os.path.join(fold_train_raw, class_name),
            dst_dir=os.path.join(fold_train_dir, class_name),
            target_count=TARGET_COUNT
        )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        fold_train_dir, shuffle=True, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

    val_ds = tf.keras.utils.image_dataset_from_directory(
        fold_val_dir, shuffle=False, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

    if os.path.exists(fold_train_raw):
        shutil.rmtree(fold_train_raw)

    return fold_dir, train_ds, val_ds

def evaluate_model(model, dataset):
    y_true, y_pred, y_proba = [], [], []
    for images, labels in dataset:
        preds = model(images, training=False).numpy()
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_proba.extend(preds)
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    y_proba = np.array(y_proba)
    acc = (y_true == y_pred).mean()
    f1 = f1_score(y_true, y_pred, average='macro')
    return acc, f1, y_true, y_pred, y_proba

NUM_CLASSES = 3

def build_metrics():
    """Tạo metrics list với per-class Precision/Recall → Macro F1 chính xác.
    Thay vì Precision()/Recall() mặc định (micro-average gộp tất cả classes),
    dùng class_id để tính riêng từng class → macro average sau."""
    metrics = ['accuracy']
    for c in range(NUM_CLASSES):
        metrics.append(tf.keras.metrics.Precision(class_id=c, name=f'prec_{c}'))
        metrics.append(tf.keras.metrics.Recall(class_id=c, name=f'rec_{c}'))
    return metrics

def extract_macro_f1_from_history(history_dict, prefix=''):
    """Tính Macro F1 per-epoch từ per-class Precision/Recall trong history.
    F1_class = 2*P*R/(P+R) cho mỗi class, rồi mean() → Macro F1.
    Khớp với sklearn.metrics.f1_score(average='macro')."""
    n_epochs = len(history_dict[f'{prefix}prec_0'])
    macro_f1s = []
    for ep in range(n_epochs):
        f1s_per_class = []
        for c in range(NUM_CLASSES):
            p = history_dict[f'{prefix}prec_{c}'][ep]
            r = history_dict[f'{prefix}rec_{c}'][ep]
            f1_c = 2 * p * r / (p + r + 1e-7)
            f1s_per_class.append(f1_c)
        macro_f1s.append(float(np.mean(f1s_per_class)))
    return macro_f1s

def plot_training_curves(fold_histories, title_prefix, save_prefix):
    """Vẽ 4 đồ thị: Loss, Accuracy, Train Macro F1, Val Macro F1 cho tất cả folds."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fold_colors = ['#1E88E5', '#43A047', '#FB8C00', '#E53935', '#8E24AA']

    # 1. Plot Loss
    ax = axes[0][0]
    for f_idx, fh in enumerate(fold_histories):
        if 'loss' not in fh: continue
        epochs = range(1, len(fh['loss']) + 1)
        c = fold_colors[f_idx % len(fold_colors)]
        ax.plot(epochs, fh['loss'], color=c, alpha=0.4, linewidth=1)
        ax.plot(epochs, fh['val_loss'], color=c, alpha=0.9, linewidth=2, label=f'Fold {f_idx+1} Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'{title_prefix} — Loss', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 2. Plot Accuracy
    ax = axes[0][1]
    for f_idx, fh in enumerate(fold_histories):
        if 'accuracy' not in fh: continue
        epochs = range(1, len(fh['accuracy']) + 1)
        c = fold_colors[f_idx % len(fold_colors)]
        ax.plot(epochs, fh['accuracy'], color=c, alpha=0.4, linewidth=1)
        ax.plot(epochs, fh['val_accuracy'], color=c, alpha=0.9, linewidth=2, label=f'Fold {f_idx+1} Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{title_prefix} — Accuracy', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Tính Macro F1 per-epoch từ per-class Precision/Recall
    all_train_f1 = []
    all_val_f1 = []
    for fh in fold_histories:
        if 'prec_0' in fh and 'rec_0' in fh:
            train_f1 = extract_macro_f1_from_history(fh, prefix='')
            val_f1 = extract_macro_f1_from_history(fh, prefix='val_')
            all_train_f1.append(train_f1)
            all_val_f1.append(val_f1)

    max_len = max([len(f1) for f1 in all_train_f1] + [0])
    if max_len > 0:
        mean_train_f1 = np.nanmean([f1 + [np.nan]*(max_len - len(f1)) for f1 in all_train_f1], axis=0)
        mean_val_f1 = np.nanmean([f1 + [np.nan]*(max_len - len(f1)) for f1 in all_val_f1], axis=0)

        # 3. Plot Train F1
        ax = axes[1][0]
        for f_idx, f1 in enumerate(all_train_f1):
            epochs = range(1, len(f1) + 1)
            c = fold_colors[f_idx % len(fold_colors)]
            ax.plot(epochs, f1, color=c, alpha=0.4, linewidth=1.5, label=f'Fold {f_idx+1}')
        ax.plot(range(1, max_len + 1), mean_train_f1, color='black', linewidth=3, label='Mean Train F1')
        ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
        ax.set_title(f'{title_prefix} — Train Macro F1-Score', fontsize=13, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # 4. Plot Val F1
        ax = axes[1][1]
        for f_idx, f1 in enumerate(all_val_f1):
            epochs = range(1, len(f1) + 1)
            c = fold_colors[f_idx % len(fold_colors)]
            ax.plot(epochs, f1, color=c, alpha=0.4, linewidth=1.5, label=f'Fold {f_idx+1}')
        ax.plot(range(1, max_len + 1), mean_val_f1, color='black', linewidth=3, label='Mean Val F1')
        ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
        ax.set_title(f'{title_prefix} — Val Macro F1-Score', fontsize=13, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_prefix}_training_curves.png", dpi=150)
    plt.show()
    plt.close('all')

print("Helper functions đã sẵn sàng.")

In [ ]:
# Cell 7: RANDOM SEARCH HEAD TRÊN TẬP TRAIN
N_TRIALS_HEAD = 7
EPOCHS_HEAD_SEARCH = 40
PATIENCE_HEAD = 10
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

head_trials = sample_head_params(N_TRIALS_HEAD, seed=SEED)
all_head_results = []
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

for trial_idx, params in enumerate(head_trials):
    print(f"\n{'='*60}")
    print(f"  Trial {trial_idx+1}/{N_TRIALS_HEAD} | LR={params['lr_phase1']:.6f} | Drop={params['dropout_rate']:.3f} | Units={params['hidden_units']}")
    print(f"{'='*60}")

    trial_fold_scores = []
    trial_best_epochs = []
    trial_fold_history = []

    for fold_idx, (train_indices, val_indices) in enumerate(
            skf.split(pool_images, pool_labels_int)):

        fold_dir, train_ds, val_ds = prepare_fold_data(fold_idx, train_indices, val_indices)

        model = build_mobilenetv2(
            num_classes=3,
            dropout_rate=params['dropout_rate'],
            hidden_units=params['hidden_units']
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', mode='max', patience=PATIENCE_HEAD,
            min_delta=0.001, restore_best_weights=True, verbose=0)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr_phase1']),
            loss='categorical_crossentropy',
            metrics=build_metrics())

        h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD_SEARCH, callbacks=[early_stop], verbose=0)

        acc, f1, _, _, _ = evaluate_model(model, val_ds)
        best_epoch = np.argmax(h.history['val_accuracy']) + 1
        fold_history = {k: list(v) for k, v in h.history.items()}

        trial_fold_scores.append({'accuracy': acc, 'f1_macro': f1})
        trial_best_epochs.append(best_epoch)
        trial_fold_history.append(fold_history)
        print(f"  Fold {fold_idx+1}: Acc={acc:.4f} | F1={f1:.4f} | BestEpoch={best_epoch}")

        del train_ds, val_ds, model, h, early_stop
        shutil.rmtree(fold_dir)
        tf.keras.backend.clear_session()
        gc.collect()

    accs = [s['accuracy'] for s in trial_fold_scores]
    f1s  = [s['f1_macro'] for s in trial_fold_scores]

    all_head_results.append({
        'trial': trial_idx + 1,
        'params': params,
        'mean_acc': np.mean(accs),
        'std_acc':  np.std(accs),
        'mean_f1':  np.mean(f1s),
        'std_f1':   np.std(f1s),
        'avg_best_epoch': np.mean(trial_best_epochs),
        'fold_best_epochs': list(trial_best_epochs),
        'fold_histories': trial_fold_history
    })
    print(f"  → Trial {trial_idx+1} Summary: F1={np.mean(f1s):.4f}±{np.std(f1s):.4f} | AvgEpoch={np.mean(trial_best_epochs):.1f}")
    del trial_fold_scores, trial_best_epochs, trial_fold_history
    gc.collect()

In [ ]:
# Cell 8: Phân Tích Kết Quả CV → Chọn Top 2 + Vẽ Đồ Thị
print("\n" + "=" * 70)
print("  KẾT QUẢ: RANDOM SEARCH HEAD (K-Fold CV)")
print("=" * 70)

df_head = pd.DataFrame([{
    'Trial': r['trial'],
    'LR': r['params']['lr_phase1'],
    'Dropout': r['params']['dropout_rate'],
    'Units': r['params']['hidden_units'],
    'Mean_Acc': r['mean_acc'],
    'Std_Acc': r['std_acc'],
    'Mean_F1': r['mean_f1'],
    'Std_F1': r['std_f1'],
    'Score': r['mean_f1'] - r['std_f1'],
    'Avg_Epoch': r['avg_best_epoch'],
} for r in all_head_results])

print(df_head.sort_values(by="Score", ascending=False).to_string(index=False))

# ── Tìm Best / Worst Trial ──
sorted_trials_asc = sorted(all_head_results, key=lambda x: x['mean_f1'])
sorted_trials_desc = sorted(
    all_head_results,
    key=lambda x: (x['mean_f1'] - x['std_f1']),
    reverse=True
)
best_trial = sorted_trials_desc[0]
worst_trial = sorted_trials_asc[0]

# ── 1. Bar Chart: F1-macro & Accuracy theo Trial ──
best_idx = next(i for i, r in enumerate(all_head_results) if r['trial'] == best_trial['trial'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['#4CAF50' if i == best_idx else '#78909C' for i in range(len(all_head_results))]
ax.bar(df_head['Trial'], df_head['Mean_F1'], color=colors,
       yerr=df_head['Std_F1'], capsize=3, alpha=0.85)
ax.axhline(y=best_trial['mean_f1'], color='#FF5722', linestyle='--', alpha=0.7,
           label=f"Best: {best_trial['mean_f1']:.4f}")
ax.set_xlabel("Trial"); ax.set_ylabel("Mean F1-macro")
ax.set_title("Head Search: F1-macro"); ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(df_head['Trial'], df_head['Mean_Acc'], color=colors,
       yerr=df_head['Std_Acc'], capsize=3, alpha=0.85)
ax.axhline(y=best_trial['mean_acc'], color='#FF5722', linestyle='--', alpha=0.7,
           label=f"Best: {best_trial['mean_acc']:.4f}")
ax.set_xlabel("Trial"); ax.set_ylabel("Mean Accuracy")
ax.set_title("Head Search: Accuracy"); ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("head_search_results.png", dpi=150)
plt.show()
plt.close('all')

# ── 2. Hyperparameter Importance: LR, Dropout, Hidden Units vs F1 ──
df_head['LR_log'] = np.log10(df_head['LR'].astype(float))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(df_head['LR_log'], df_head['Mean_F1'], c=df_head['Mean_F1'],
           cmap='RdYlGn', s=100, edgecolors='black', zorder=5)
ax.scatter(np.log10(best_trial['params']['lr_phase1']), best_trial['mean_f1'],
           color='blue', s=300, marker='*', edgecolors='black', label='Best', zorder=10)
ax.set_xlabel("Learning Rate (log10)"); ax.set_ylabel("Mean F1-macro")
ax.set_title("Learning Rate vs F1-macro"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(df_head['Dropout'], df_head['Mean_F1'], c=df_head['Mean_F1'],
           cmap='RdYlGn', s=100, edgecolors='black', zorder=5)
ax.scatter(best_trial['params']['dropout_rate'], best_trial['mean_f1'],
           color='blue', s=300, marker='*', edgecolors='black', label='Best', zorder=10)
ax.set_xlabel("Dropout Rate"); ax.set_ylabel("Mean F1-macro")
ax.set_title("Dropout Rate vs F1-macro"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
ax.scatter(df_head['Units'], df_head['Mean_F1'], c=df_head['Mean_F1'],
           cmap='RdYlGn', s=100, edgecolors='black', zorder=5)
ax.scatter(best_trial['params']['hidden_units'], best_trial['mean_f1'],
           color='blue', s=300, marker='*', edgecolors='black', label='Best', zorder=10)
ax.set_xlabel("Hidden Units"); ax.set_ylabel("Mean F1-macro")
ax.set_title("Hidden Units vs F1-macro"); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Hyperparameter Importance — Vì sao chọn Best Trial?", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("hyperparameters_vs_f1.png", dpi=150)
plt.show()
plt.close('all')

# ── 3. Training Curves: Best Trial vs Worst Trial ──
print("\n" + "=" * 70)
print(f"  TRAINING CURVES — BEST TRIAL {best_trial['trial']} "
      f"(F1={best_trial['mean_f1']:.4f}±{best_trial['std_f1']:.4f})")
print("=" * 70)
plot_training_curves(best_trial['fold_histories'],
                     f"Trial {best_trial['trial']} (Best)", "head_best")

print("\n" + "=" * 70)
print(f"  TRAINING CURVES — WORST TRIAL {worst_trial['trial']} "
      f"(F1={worst_trial['mean_f1']:.4f}±{worst_trial['std_f1']:.4f})")
print("=" * 70)
plot_training_curves(worst_trial['fold_histories'],
                     f"Trial {worst_trial['trial']} (Worst)", "head_worst")

# ── B1: CV Epoch Boxplot — Phân bố Best Epoch qua 5 Folds ──
fig, ax = plt.subplots(figsize=(12, 6))
box_data = [r['fold_best_epochs'] for r in all_head_results]
trial_labels = [f"Trial {r['trial']}" for r in all_head_results]

best_trial_idx_bp = next(i for i, r in enumerate(all_head_results) if r['trial'] == best_trial['trial'])
bp_colors = ['#4CAF50' if i == best_trial_idx_bp else '#78909C' for i in range(len(all_head_results))]

bp = ax.boxplot(box_data, labels=trial_labels, patch_artist=True,
                boxprops=dict(linewidth=1.5),
                medianprops=dict(color='#D32F2F', linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, epochs_list in enumerate(box_data):
    x = np.random.normal(i + 1, 0.04, size=len(epochs_list))
    ax.scatter(x, epochs_list, alpha=0.8, color='#1565C0', s=40, zorder=5, edgecolors='white')

ax.set_xlabel('Trial', fontsize=12)
ax.set_ylabel('Best Epoch', fontsize=12)
ax.set_title('K-Fold CV — Phân Bố Best Epoch Qua 5 Folds', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("cv_epoch_boxplot.png", dpi=150)
plt.show()
plt.close('all')

# ── Chọn Top 2 Trials ──
top_2_trials = sorted_trials_desc[:2]

print(f"\n{'='*70}")
print("  TOP 2 TRIALS TỐT NHẤT TỪ K-FOLD → TRADITIONAL TRAINING")
print(f"{'='*70}")
for i, t in enumerate(top_2_trials):
    print(f"Top {i+1} - Trial {t['trial']}: F1={t['mean_f1']:.4f}±{t['std_f1']:.4f} | "
          f"Avg Epoch={t['avg_best_epoch']:.1f}")
    print(f"   Tham số: LR={t['params']['lr_phase1']:.6f}, "
          f"Dropout={t['params']['dropout_rate']:.3f}, "
          f"Units={t['params']['hidden_units']}")

gc.collect()

In [ ]:
# Cell 9: LƯU CHECKPOINT HEAD VỀ DRIVE
import json

CHECKPOINT_DIR = "/content/drive/MyDrive/PBL5/Checkpoint/cashew_model_mobilenetv2_holdout_v3/checkpoints_head"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

head_checkpoint = {
    'all_head_results': [{
        'trial': r['trial'],
        'params': r['params'],
        'mean_acc': r['mean_acc'],
        'std_acc': r['std_acc'],
        'mean_f1': r['mean_f1'],
        'std_f1': r['std_f1'],
        'avg_best_epoch': r['avg_best_epoch'],
        'fold_best_epochs': r.get('fold_best_epochs', []),
    } for r in all_head_results],
    'top_2_trials': [{
        'trial': t['trial'],
        'params': t['params'],
        'mean_acc': t['mean_acc'],
        'std_acc': t['std_acc'],
        'mean_f1': t['mean_f1'],
        'std_f1': t['std_f1'],
        'avg_best_epoch': t['avg_best_epoch'],
    } for t in top_2_trials],
    'config': {
        'N_TRIALS_HEAD': N_TRIALS_HEAD,
        'EPOCHS_HEAD_SEARCH': EPOCHS_HEAD_SEARCH,
        'PATIENCE_HEAD': PATIENCE_HEAD,
        'K_FOLDS': K_FOLDS,
        'SEED': SEED,
        'TARGET_COUNT': TARGET_COUNT,
        'IMG_SIZE': IMG_SIZE,
        'BATCH_SIZE': BATCH_SIZE
    }
}

checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_head_search.json")

# Hàm hỗ trợ chuyển đổi kiểu dữ liệu NumPy sang Python thuần
def numpy_converter(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

with open(checkpoint_path, 'w') as f:
    json.dump(head_checkpoint, f, indent=2, default=numpy_converter)

for chart in ["head_search_results.png", "hyperparameters_vs_f1.png",
              "head_best_training_curves.png", "head_worst_training_curves.png",
              "cv_epoch_boxplot.png"]:
    if os.path.exists(chart):
        shutil.copy(chart, os.path.join(CHECKPOINT_DIR, chart))

print(f"Checkpoint đã lưu tại: {checkpoint_path}")
print("→ Restart runtime rồi chạy Cell 10 để load checkpoint.")

In [ ]:
# Cell 10: LOAD CHECKPOINT HEAD (sau khi restart runtime)
# ═══════════════════════════════════════════════════════════
# Chạy lại Cell 1 → 6 (import, crop, config, augment, model, helpers)
# rồi chạy cell này để load checkpoint từ Drive
# ═══════════════════════════════════════════════════════════
import json

CHECKPOINT_DIR = "/content/drive/MyDrive/PBL5/Checkpoint/cashew_model_mobilenetv2_holdout_v3/checkpoints_head"
checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_head_search.json")

if not os.path.exists(checkpoint_path):
    print(f"Không tìm thấy checkpoint tại: {checkpoint_path}")
    print("   Bạn cần chạy Cell 7 → 9 trước.")
else:
    with open(checkpoint_path, 'r') as f:
        head_checkpoint = json.load(f)

    all_head_results = head_checkpoint['all_head_results']
    top_2_trials = head_checkpoint['top_2_trials']

    N_TRIALS_HEAD = head_checkpoint['config']['N_TRIALS_HEAD']
    EPOCHS_HEAD_SEARCH = head_checkpoint['config']['EPOCHS_HEAD_SEARCH']
    PATIENCE_HEAD = head_checkpoint['config']['PATIENCE_HEAD']
    SEED = head_checkpoint['config']['SEED']
    TARGET_COUNT = head_checkpoint['config']['TARGET_COUNT']
    IMG_SIZE = tuple(head_checkpoint['config']['IMG_SIZE'])
    BATCH_SIZE = head_checkpoint['config']['BATCH_SIZE']

    print(f"Đã load checkpoint thành công!")
    print(f"   Số trials: {len(all_head_results)}")
    print(f"\n{'='*70}")
    print("  TOP 2 TRIALS TỪ K-FOLD")
    print(f"{'='*70}")
    for i, t in enumerate(top_2_trials):
        print(f"Top {i+1} - Trial {t['trial']}: F1={t['mean_f1']:.4f}±{t['std_f1']:.4f} | "
              f"Avg Epoch={t['avg_best_epoch']:.1f}")
        print(f"   Tham số: {t['params']}")
    print(f"\n→ Sẵn sàng chạy Traditional Training (Cell 11).")

In [ ]:
# Cell 11: Chuẩn Bị Data Truyền Thống (Train 78% / Valid 11%)
random.seed(SEED)
print("Chuẩn bị Train Dataset và Valid Dataset cho Traditional Training...")

trad_train_dir = "/content/trad_train"
trad_val_dir = "/content/trad_val"
trad_train_raw = "/content/trad_train_raw"

if os.path.exists(trad_train_dir): shutil.rmtree(trad_train_dir)
if os.path.exists(trad_val_dir): shutil.rmtree(trad_val_dir)
if os.path.exists(trad_train_raw): shutil.rmtree(trad_train_raw)

# 1. Tập Valid (11%) -> Không Augment
for img_path, label in zip(valid_images, valid_labels):
    dst = os.path.join(trad_val_dir, label)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(img_path, dst)

# 2. Tập Train (78%) -> Copy vào Raw rồi Augment
for img_path, label in zip(train_images, train_labels):
    dst = os.path.join(trad_train_raw, label)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(img_path, dst)

for class_name in class_mapping.values():
    augment(
        src_dir=os.path.join(trad_train_raw, class_name),
        dst_dir=os.path.join(trad_train_dir, class_name),
        target_count=TARGET_COUNT
    )

train_ds_trad = tf.keras.utils.image_dataset_from_directory(
    trad_train_dir, shuffle=True, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds_trad = tf.keras.utils.image_dataset_from_directory(
    trad_val_dir, shuffle=False, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Cell 12: Traditional Training 2 Trials + So Sánh Epoch CV vs Actual
trad_results = []
best_val_f1_overall = -1
best_trial_info = None

for rank, trial in enumerate(top_2_trials):
    print(f"\n{'='*70}")
    print(f"  TRADITIONAL TRAINING — TOP {rank+1} (Trial {trial['trial']})")
    print(f"  Avg Epoch từ CV: {trial['avg_best_epoch']:.1f}")
    print(f"{'='*70}")

    params = trial['params']
    model = build_mobilenetv2(num_classes=3, dropout_rate=params['dropout_rate'],
                              hidden_units=params['hidden_units'])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr_phase1']),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )

    early_stop_trad = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', mode='max', patience=15,
        restore_best_weights=True, verbose=1
    )

    h = model.fit(train_ds_trad, validation_data=val_ds_trad, epochs=50, callbacks=[early_stop_trad])

    actual_best_epoch = np.argmax(h.history['val_accuracy']) + 1
    val_acc, val_f1, _, _, _ = evaluate_model(model, val_ds_trad)

    trad_results.append({
        'trial': trial['trial'],
        'params': params,
        'cv_avg_epoch': trial['avg_best_epoch'],
        'actual_best_epoch': actual_best_epoch,
        'val_acc': val_acc,
        'val_f1': val_f1,
        'cv_mean_f1': trial['mean_f1'],
        'cv_std_f1': trial['std_f1'],
        'history': {k: list(v) for k, v in h.history.items()},
    })

    print(f"\nKết Quả Trial {trial['trial']}:")
    print(f"  - Avg Epoch (CV K=5): {trial['avg_best_epoch']:.1f}")
    print(f"  - Actual Best Epoch (Truyền thống): {actual_best_epoch}")
    print(f"  - Valid Accuracy: {val_acc:.4f} | Valid F1: {val_f1:.4f}")

    if val_f1 > best_val_f1_overall:
        best_val_f1_overall = val_f1
        best_trial_info = trad_results[-1]
        model.save("best_trad_mobilenetv2_cashew.keras")
        print("  -> Lưu model này làm model Tốt Nhất.")

    del model, h
    tf.keras.backend.clear_session()
    gc.collect()

print("\nĐã huấn luyện xong. Bạn có thể đối chiếu Epoch thực tế với Epoch CV.")

# ── B2a: Traditional Training Curves — Loss, Accuracy, Macro F1 cho mỗi Trial ──
for r in trad_results:
    hist = r['history']
    trial_id = r['trial']
    best_ep = r['actual_best_epoch']
    n_epochs = len(hist['loss'])
    epochs_range = range(1, n_epochs + 1)

    # Tính Macro F1 per-epoch từ per-class metrics
    train_macro_f1 = extract_macro_f1_from_history(hist, prefix='')
    val_macro_f1 = extract_macro_f1_from_history(hist, prefix='val_')

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # 1. Loss
    ax = axes[0]
    ax.plot(epochs_range, hist['loss'], color='#1E88E5', linewidth=2, alpha=0.6, label='Train Loss')
    ax.plot(epochs_range, hist['val_loss'], color='#E53935', linewidth=2.5, label='Val Loss')
    ax.axvline(x=best_ep, color='#4CAF50', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'Best Epoch = {best_ep}')
    ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Trial {trial_id} — Loss', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # 2. Accuracy
    ax = axes[1]
    ax.plot(epochs_range, hist['accuracy'], color='#1E88E5', linewidth=2, alpha=0.6, label='Train Acc')
    ax.plot(epochs_range, hist['val_accuracy'], color='#E53935', linewidth=2.5, label='Val Acc')
    ax.axvline(x=best_ep, color='#4CAF50', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'Best Epoch = {best_ep}')
    ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Trial {trial_id} — Accuracy', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # 3. Macro F1
    ax = axes[2]
    ax.plot(epochs_range, train_macro_f1, color='#1E88E5', linewidth=2, alpha=0.6, label='Train Macro F1')
    ax.plot(epochs_range, val_macro_f1, color='#E53935', linewidth=2.5, label='Val Macro F1')
    ax.axvline(x=best_ep, color='#4CAF50', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'Best Epoch = {best_ep}')
    ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title(f'Trial {trial_id} — Macro F1-Score', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    plt.suptitle(f'Traditional Training — Trial {trial_id} '
                 f'(LR={r["params"]["lr_phase1"]:.6f}, Drop={r["params"]["dropout_rate"]:.3f}, '
                 f'Units={r["params"]["hidden_units"]})',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"trad_trial_{trial_id}_training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
    plt.close('all')
    print(f"  Trial {trial_id}: Best Epoch={best_ep} | "
          f"Val F1(macro)={val_macro_f1[best_ep-1]:.4f} | Val Acc={hist['val_accuracy'][best_ep-1]:.4f}")

# ── B2b: Traditional vs CV Epoch Comparison ──
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(trad_results))
bar_width = 0.3

cv_epochs = [r['cv_avg_epoch'] for r in trad_results]
actual_epochs = [r['actual_best_epoch'] for r in trad_results]
trial_names = [f"Trial {r['trial']}" for r in trad_results]

bars_cv = ax.bar(x_pos - bar_width/2, cv_epochs, bar_width,
                 label='CV Avg Epoch', color='#1E88E5', alpha=0.85, edgecolor='white')
bars_actual = ax.bar(x_pos + bar_width/2, actual_epochs, bar_width,
                     label='Actual Best Epoch', color='#FB8C00', alpha=0.85, edgecolor='white')

for i, (cv_e, act_e) in enumerate(zip(cv_epochs, actual_epochs)):
    pct_diff = ((act_e - cv_e) / cv_e) * 100
    ax.annotate(f'{pct_diff:+.1f}%', xy=(i + bar_width/2, act_e),
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color='#D32F2F' if abs(pct_diff) > 20 else '#2E7D32')

    ax.fill_between([i - 0.45, i + 0.45], cv_e * 0.8, cv_e * 1.2,
                    alpha=0.08, color='#4CAF50', zorder=0)

ax.set_xticks(x_pos)
ax.set_xticklabels(trial_names, fontsize=12)
ax.set_xlabel('Trial', fontsize=12)
ax.set_ylabel('Best Epoch', fontsize=12)
ax.set_title('Traditional Training vs K-Fold CV — So Sánh Optimal Epoch',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars_cv:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)
for bar in bars_actual:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("trad_vs_cv_epoch.png", dpi=150)
plt.show()
plt.close('all')

BEST_TRAD_PARAMS = best_trial_info['params']
BEST_TRAD_EPOCHS = int(np.ceil(best_trial_info['actual_best_epoch'] * 1.2))

In [ ]:
# Cell 13: HUẤN LUYỆN FINAL MODEL (78% train + 11% valid)
FINAL_LR       = BEST_TRAD_PARAMS['lr_phase1']
FINAL_DROPOUT  = BEST_TRAD_PARAMS['dropout_rate']
FINAL_HIDDEN   = BEST_TRAD_PARAMS['hidden_units']
FINAL_EPOCHS   = BEST_TRAD_EPOCHS

print(f"\n{'='*60}")
print(f"  HUẤN LUYỆN FINAL MODEL (FREEZE BACKBONE)")
print(f"{'='*60}")
print(f"  LR: {FINAL_LR:.6f} | Epochs: {FINAL_EPOCHS}")
print(f"  Dropout: {FINAL_DROPOUT:.3f} | Units: {FINAL_HIDDEN} | SE: 16 (cố định)")
print(f"{'='*60}")

final_images = pool_images
final_labels = pool_labels

final_train_dir = "/content/final_train"
final_train_raw = "/content/final_train_raw"
if os.path.exists(final_train_dir): shutil.rmtree(final_train_dir)
if os.path.exists(final_train_raw): shutil.rmtree(final_train_raw)

for img_path, label in zip(final_images, final_labels):
    dst = os.path.join(final_train_raw, label)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(img_path, dst)

for class_name in class_mapping.values():
    augment(
        src_dir=os.path.join(final_train_raw, class_name),
        dst_dir=os.path.join(final_train_dir, class_name),
        target_count=TARGET_COUNT
    )

final_train_ds = tf.keras.utils.image_dataset_from_directory(
    final_train_dir, shuffle=True, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

final_model = build_mobilenetv2(
    num_classes=3, dropout_rate=FINAL_DROPOUT, hidden_units=FINAL_HIDDEN
)

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')])

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

final_h1 = final_model.fit(final_train_ds, epochs=FINAL_EPOCHS,
                            callbacks=[reduce_lr], verbose=1)

final_model.save("final_mobilenetv2_cashew.keras")
print("\nFinal Model đã lưu: final_mobilenetv2_cashew.keras")

# ── A2: Final Model Training Curves ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs_range = range(1, len(final_h1.history['loss']) + 1)

# Loss
ax = axes[0]
ax.plot(epochs_range, final_h1.history['loss'], color='#1E88E5', linewidth=2, label='Training Loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Final Model — Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
final_loss = final_h1.history['loss'][-1]
ax.annotate(f'{final_loss:.4f}', xy=(len(final_h1.history['loss']), final_loss),
            fontsize=10, fontweight='bold', color='#1565C0',
            xytext=(-40, 10), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#1565C0'))

# Accuracy
ax = axes[1]
ax.plot(epochs_range, final_h1.history['accuracy'], color='#43A047', linewidth=2, label='Training Accuracy')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Final Model — Training Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
final_acc = final_h1.history['accuracy'][-1]
ax.annotate(f'{final_acc:.4f}', xy=(len(final_h1.history['accuracy']), final_acc),
            fontsize=10, fontweight='bold', color='#2E7D32',
            xytext=(-40, -15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#2E7D32'))

plt.suptitle(f'Final Model Training Curves (Epochs={FINAL_EPOCHS}, LR={FINAL_LR:.6f})',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("final_model_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

shutil.rmtree(final_train_dir)
shutil.rmtree(final_train_raw)
del final_h1

In [ ]:
# Cell 14: ĐÁNH GIÁ FINAL MODEL TRÊN TẬP TEST (11%)
print(f"\n{'='*60}")
print(f"  ĐÁNH GIÁ FINAL MODEL TRÊN TẬP TEST (11%)")
print(f"{'='*60}")

test_eval_dir = "/content/test_eval_dir"
if os.path.exists(test_eval_dir): shutil.rmtree(test_eval_dir)

for img_path, label in zip(test_images, test_labels):
    dst = os.path.join(test_eval_dir, label)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(img_path, dst)

test_ds_eval = tf.keras.utils.image_dataset_from_directory(
    test_eval_dir, shuffle=False, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

final_model_keras = tf.keras.models.load_model("final_mobilenetv2_cashew.keras")
keras_test_acc, keras_test_f1, y_true_keras, y_pred_keras, y_proba_keras = \
    evaluate_model(final_model_keras, test_ds_eval)

classes_name = ["broken", "defect", "whole"]

print(f"\n  Keras — Accuracy: {keras_test_acc:.4f} | F1-macro: {keras_test_f1:.4f}")
print("\nClassification Report (Keras):")
print(classification_report(y_true_keras, y_pred_keras, target_names=classes_name, digits=4))

cm_keras = confusion_matrix(y_true_keras, y_pred_keras)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sns.heatmap(cm_keras, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes_name, yticklabels=classes_name, ax=ax,
            annot_kws={"size": 14, "fontweight": "bold"})
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Ground Truth", fontsize=12)
ax.set_title("Confusion Matrix (Count)", fontsize=14, fontweight='bold')

cm_normalized = cm_keras.astype('float') / cm_keras.sum(axis=1)[:, np.newaxis]
ax = axes[1]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Oranges',
            xticklabels=classes_name, yticklabels=classes_name, ax=ax,
            annot_kws={"size": 14, "fontweight": "bold"})
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Ground Truth", fontsize=12)
ax.set_title("Confusion Matrix (Normalized)", fontsize=14, fontweight='bold')

plt.suptitle("Final Model — Confusion Matrix trên Tập Test 11%",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("final_model_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# ── A1: Per-Class F1 Bar Chart ──
from sklearn.metrics import precision_score, recall_score

per_class_p = precision_score(y_true_keras, y_pred_keras, average=None)
per_class_r = recall_score(y_true_keras, y_pred_keras, average=None)
per_class_f1 = f1_score(y_true_keras, y_pred_keras, average=None)

fig, ax = plt.subplots(figsize=(12, 6))
x_cls = np.arange(len(classes_name))
w = 0.25

bars_p = ax.bar(x_cls - w, per_class_p, w, label='Precision', color='#1E88E5', alpha=0.85, edgecolor='white')
bars_r = ax.bar(x_cls, per_class_r, w, label='Recall', color='#FB8C00', alpha=0.85, edgecolor='white')
bars_f = ax.bar(x_cls + w, per_class_f1, w, label='F1-Score', color='#E53935', alpha=0.85, edgecolor='white')

ax.axhline(y=keras_test_f1, color='#4CAF50', linestyle='--', linewidth=2, alpha=0.8,
           label=f'Macro F1 = {keras_test_f1:.4f}')

for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x_cls)
ax.set_xticklabels(classes_name, fontsize=12)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_title('Final Model — Precision / Recall / F1 Per Class (Test Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("per_class_f1_bar.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# ── C1: Precision-Recall Curve (Multi-class, One-vs-Rest) ──
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score

y_true_bin = label_binarize(y_true_keras, classes=[0, 1, 2])
y_proba_np = np.array(y_proba_keras)

fig, ax = plt.subplots(figsize=(10, 8))
pr_colors = ['#1E88E5', '#FB8C00', '#E53935']

for i, (cls_name, color) in enumerate(zip(classes_name, pr_colors)):
    precision_vals, recall_vals, _ = precision_recall_curve(y_true_bin[:, i], y_proba_np[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_proba_np[:, i])
    ax.plot(recall_vals, precision_vals, color=color, linewidth=2,
            label=f'{cls_name} (AP={ap:.3f})')

precision_micro, recall_micro, _ = precision_recall_curve(y_true_bin.ravel(), y_proba_np.ravel())
ap_micro = average_precision_score(y_true_bin, y_proba_np, average='micro')
ax.plot(recall_micro, precision_micro, color='black', linewidth=2.5, linestyle='--',
        label=f'Micro-Avg (AP={ap_micro:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Final Model — Precision-Recall Curve (One-vs-Rest)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower left')
ax.set_xlim([0, 1.02])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("precision_recall_curve.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# ── C2: Confidence Distribution Histogram ──
max_proba = np.max(y_proba_np, axis=1)
correct_mask = (y_true_keras == y_pred_keras)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram: Correct vs Incorrect
ax = axes[0]
ax.hist(max_proba[correct_mask], bins=20, alpha=0.7, color='#43A047',
        label=f'Correct ({correct_mask.sum()})', edgecolor='white')
ax.hist(max_proba[~correct_mask], bins=20, alpha=0.7, color='#E53935',
        label=f'Incorrect ({(~correct_mask).sum()})', edgecolor='white')
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='Threshold 0.5')
ax.axvline(x=0.9, color='#FF9800', linestyle='--', alpha=0.7, label='Threshold 0.9')
ax.set_xlabel('Max Prediction Confidence', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Confidence Distribution — Correct vs Incorrect', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Calibration-like: Per-bin accuracy
ax = axes[1]
n_bins_cal = 10
bin_boundaries = np.linspace(0, 1, n_bins_cal + 1)
bin_accs = []
bin_confs = []
bin_counts = []
for b in range(n_bins_cal):
    mask = (max_proba >= bin_boundaries[b]) & (max_proba < bin_boundaries[b+1])
    if mask.sum() > 0:
        bin_accs.append(correct_mask[mask].mean())
        bin_confs.append(max_proba[mask].mean())
        bin_counts.append(mask.sum())
    else:
        bin_accs.append(0)
        bin_confs.append((bin_boundaries[b] + bin_boundaries[b+1]) / 2)
        bin_counts.append(0)

bin_centers = [(bin_boundaries[i] + bin_boundaries[i+1]) / 2 for i in range(n_bins_cal)]
ax.bar(bin_centers, bin_accs, width=0.08, alpha=0.7, color='#1E88E5', edgecolor='white', label='Accuracy in Bin')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect Calibration')
ax.set_xlabel('Mean Predicted Confidence', fontsize=12)
ax.set_ylabel('Fraction Correct', fontsize=12)
ax.set_title('Reliability Diagram (Calibration)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)

plt.suptitle('Final Model — Confidence Analysis (Test Set)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("confidence_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

shutil.rmtree(test_eval_dir)

In [ ]:
# Cell 15: CHUYỂN ĐỔI KERAS → TFLITE
converter = tf.lite.TFLiteConverter.from_keras_model(final_model_keras)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Chỉ BUILTINS — portable, nhẹ hơn
]

tflite_model = converter.convert()

tflite_model_path = "final_mobilenetv2_cashew.tflite"
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

keras_size_mb = os.path.getsize("final_mobilenetv2_cashew.keras") / (1024 * 1024)
tflite_size_mb = os.path.getsize(tflite_model_path) / (1024 * 1024)

print(f"\nKeras:  {keras_size_mb:.2f} MB | TFLite: {tflite_size_mb:.2f} MB")
print(f"Nén:    {(1 - tflite_size_mb / keras_size_mb) * 100:.1f}%")

# Lưu cả 2 model lên Drive
drive_model_dir = "/content/drive/MyDrive/PBL5/Model/cashew_model_mobilenetv2_v2"
os.makedirs(drive_model_dir, exist_ok=True)
shutil.copy("final_mobilenetv2_cashew.keras",
            os.path.join(drive_model_dir, "final_mobilenetv2_cashew.keras"))
shutil.copy(tflite_model_path, os.path.join(drive_model_dir, tflite_model_path))
print(f"\nModel đã lưu lên Drive: {drive_model_dir}")

In [ ]:
# Cell 16: ĐÁNH GIÁ TFLITE MODEL TRÊN TẬP TEST (11%)
def evaluate_tflite_model(tflite_path, dataset):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_true_all, y_pred_all, y_proba_all = [], [], []
    for images, labels in dataset:
        for i in range(images.shape[0]):
            img_input = tf.expand_dims(images[i], axis=0).numpy()
            interpreter.set_tensor(input_details[0]['index'], img_input.astype(np.float32))
            interpreter.invoke()
            output_data = interpreter.get_tensor(output_details[0]['index'])
            y_true_all.append(np.argmax(labels[i].numpy()))
            y_pred_all.append(np.argmax(output_data[0]))
            y_proba_all.append(output_data[0])

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    acc = (y_true_all == y_pred_all).mean()
    f1 = f1_score(y_true_all, y_pred_all, average='macro')
    return acc, f1, y_true_all, y_pred_all, np.array(y_proba_all)

test_dir_tflite = "/content/test_eval_tflite"
if os.path.exists(test_dir_tflite): shutil.rmtree(test_dir_tflite)
for img_path, label in zip(test_images, test_labels):
    dst = os.path.join(test_dir_tflite, label)
    os.makedirs(dst, exist_ok=True)
    shutil.copy(img_path, dst)

test_ds_tflite = tf.keras.utils.image_dataset_from_directory(
    test_dir_tflite, shuffle=False, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="categorical").prefetch(buffer_size=tf.data.AUTOTUNE)

tflite_test_acc, tflite_test_f1, y_true_tflite, y_pred_tflite, y_proba_tflite = \
    evaluate_tflite_model("final_mobilenetv2_cashew.tflite", test_ds_tflite)

print(f"\n  TFLite — Accuracy: {tflite_test_acc:.4f} | F1-macro: {tflite_test_f1:.4f}")
print(classification_report(y_true_tflite, y_pred_tflite, target_names=classes_name, digits=4))

shutil.rmtree(test_dir_tflite)

In [ ]:
# Cell 17: BENCHMARKING KERAS vs TFLITE — CÙNG CPU (FAIR COMPARISON)
# ═══════════════════════════════════════════════════════════════
# QUAN TRỌNG: Disable GPU để cả Keras lẫn TFLite đều chạy CPU
# → So sánh latency/throughput mới có ý nghĩa.
# ═══════════════════════════════════════════════════════════════
import time
import psutil
# Bỏ dòng tf.config.set_visible_devices vì TF đã được init ở các cell trước

N_RUNS = 100
process = psutil.Process(os.getpid())

# ── 1. KERAS BENCHMARK ──
print("\n── Keras Benchmark (CPU) ──")
tf.keras.backend.clear_session()
gc.collect()
time.sleep(1)

ram_before_keras = process.memory_info().rss / (1024 * 1024)

# Ép Keras chạy trên CPU bằng context manager
with tf.device('/CPU:0'):
    keras_bench = tf.keras.models.load_model("final_mobilenetv2_cashew.keras")
    dummy_input_keras = np.random.rand(1, 224, 224, 3).astype(np.float32)
    for _ in range(10): _ = keras_bench(dummy_input_keras, training=False)  # Warm-up

    ram_after_keras = process.memory_info().rss / (1024 * 1024)
    ram_keras_delta = ram_after_keras - ram_before_keras
    cpu_keras_percent = psutil.cpu_percent(interval=2)

    times_single_keras = []
    for _ in range(N_RUNS):
        start = time.perf_counter()
        _ = keras_bench(dummy_input_keras, training=False)
        times_single_keras.append((time.perf_counter() - start) * 1000)

    avg_latency_keras = np.mean(times_single_keras)
    std_latency_keras = np.std(times_single_keras)

    batch_input_keras = np.random.rand(32, 224, 224, 3).astype(np.float32)
    times_batch_keras = []
    for _ in range(N_RUNS // 4):
        start = time.perf_counter()
        _ = keras_bench(batch_input_keras, training=False)
        times_batch_keras.append((time.perf_counter() - start) * 1000)
    throughput_keras = 32 * 1000.0 / np.mean(times_batch_keras)

del keras_bench, dummy_input_keras, batch_input_keras
tf.keras.backend.clear_session()
gc.collect()
time.sleep(2)

# ── 2. TFLITE BENCHMARK ──
print("── TFLite Benchmark (CPU) ──")
ram_before_tflite = process.memory_info().rss / (1024 * 1024)
interpreter = tf.lite.Interpreter(model_path="final_mobilenetv2_cashew.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

dummy_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

ram_after_tflite = process.memory_info().rss / (1024 * 1024)
ram_tflite_delta = ram_after_tflite - ram_before_tflite
cpu_tflite_percent = psutil.cpu_percent(interval=2)

times_single_tflite = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    times_single_tflite.append((time.perf_counter() - start) * 1000)

avg_latency_tflite = np.mean(times_single_tflite)
std_latency_tflite = np.std(times_single_tflite)

interpreter.resize_tensor_input(input_details[0]['index'], [32, 224, 224, 3])
interpreter.allocate_tensors()
batch_input_tflite = np.random.rand(32, 224, 224, 3).astype(np.float32)
times_batch_tflite = []
for _ in range(N_RUNS // 4):
    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], batch_input_tflite)
    interpreter.invoke()
    times_batch_tflite.append((time.perf_counter() - start) * 1000)
throughput_tflite = 32 * 1000.0 / np.mean(times_batch_tflite)

del interpreter, dummy_input, batch_input_tflite
gc.collect()

# ── 3. BẢNG SO SÁNH ──
acc_diff = (tflite_test_acc - keras_test_acc) * 100
f1_diff = (tflite_test_f1 - keras_test_f1) * 100
acc_note = f'{acc_diff:+.2f}% (≈ noise)' if abs(acc_diff) < 1.0 else f'{acc_diff:+.2f}%'
f1_note = f'{f1_diff:+.2f}% (≈ noise)' if abs(f1_diff) < 1.0 else f'{f1_diff:+.2f}%'

comparison_data = {
    'Chỉ Số': ['Model Size (MB)', 'Latency — Single (ms)',
               'Throughput — Batch=32 (fps)', 'Accuracy', 'F1-macro',
               'RAM Usage — Delta (MB)', 'CPU Usage (%)'],
    'Keras': [f'{keras_size_mb:.2f}', f'{avg_latency_keras:.2f} ± {std_latency_keras:.2f}',
              f'{throughput_keras:.1f}', f'{keras_test_acc:.4f}', f'{keras_test_f1:.4f}',
              f'{ram_keras_delta:.0f}', f'{cpu_keras_percent:.1f}'],
    'TFLite': [f'{tflite_size_mb:.2f}', f'{avg_latency_tflite:.2f} ± {std_latency_tflite:.2f}',
               f'{throughput_tflite:.1f}', f'{tflite_test_acc:.4f}', f'{tflite_test_f1:.4f}',
               f'{ram_tflite_delta:.0f}', f'{cpu_tflite_percent:.1f}'],
    'Cải Thiện': [f'{(1 - tflite_size_mb/keras_size_mb)*100:.1f}% nhẹ hơn',
                  f'{(1 - avg_latency_tflite/avg_latency_keras)*100:+.1f}%',
                  f'{(throughput_tflite/throughput_keras - 1)*100:+.1f}%',
                  acc_note, f1_note,
                  f'{(1 - ram_tflite_delta/max(ram_keras_delta, 1))*100:+.1f}%',
                  f'{(cpu_tflite_percent - cpu_keras_percent):+.1f}%']
}

df_comparison = pd.DataFrame(comparison_data)
print(f"\n{'='*90}")
print(f"  BẢNG SO SÁNH: KERAS vs TFLITE (CÙNG CPU — FAIR COMPARISON)")
print(f"{'='*90}")
print(df_comparison.to_string(index=False))
print(f"{'='*90}")
print(f"\n  GHI CHÚ:")
print(f"  * Cả Keras và TFLite đều chạy CPU (tf.config.set_visible_devices([], 'GPU'))")
print(f"  * Accuracy/F1: Chênh lệch < 1% nằm trong noise thống kê (Dynamic Range Quantization)")
print(f"  * RAM Delta: Đo lượng RAM tăng thêm khi load model (không phải tổng RSS)")

# ── C4: Benchmark Radar Chart — Keras vs TFLite ──
categories = ['Model Size\n(nhỏ hơn = tốt)', 'Latency\n(thấp hơn = tốt)',
              'Throughput\n(cao hơn = tốt)', 'Accuracy', 'F1-macro',
              'RAM\n(ít hơn = tốt)']

max_size = max(keras_size_mb, tflite_size_mb)
max_lat = max(avg_latency_keras, avg_latency_tflite)
max_thru = max(throughput_keras, throughput_tflite)
max_ram = max(abs(ram_keras_delta), abs(ram_tflite_delta), 1)

keras_vals = [
    1 - keras_size_mb / max_size,
    1 - avg_latency_keras / max_lat,
    throughput_keras / max_thru,
    keras_test_acc,
    keras_test_f1,
    1 - abs(ram_keras_delta) / max_ram
]
tflite_vals = [
    1 - tflite_size_mb / max_size,
    1 - avg_latency_tflite / max_lat,
    throughput_tflite / max_thru,
    tflite_test_acc,
    tflite_test_f1,
    1 - abs(ram_tflite_delta) / max_ram
]

N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
keras_vals += keras_vals[:1]
tflite_vals += tflite_vals[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

ax.plot(angles, keras_vals, 'o-', linewidth=2.5, color='#1E88E5', label='Keras', markersize=7)
ax.fill(angles, keras_vals, alpha=0.15, color='#1E88E5')

ax.plot(angles, tflite_vals, 'o-', linewidth=2.5, color='#FB8C00', label='TFLite', markersize=7)
ax.fill(angles, tflite_vals, alpha=0.15, color='#FB8C00')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title('Benchmark Radar — Keras vs TFLite (CPU)\n', fontsize=15, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=12)
ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig("benchmark_radar_chart.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')